# CS2: Baltic Countries Euro Adoption - Temporal Volatility Analysis

**Purpose**: Analyze changes in capital flow volatility before and after Euro adoption for Estonia, Latvia, and Lithuania.

**Source Extraction**:
- `cs2_shared_functions.py` lines 33-61 (Euro adoption dates)
- `cs2_shared_functions.py` lines 212-250 (temporal F-tests)
- `cs2_shared_functions.py` lines 165-189 (temporal statistics)

**Key Dates**:
- Estonia: Adopted Euro on January 1, 2011
- Latvia: Adopted Euro on January 1, 2014
- Lithuania: Adopted Euro on January 1, 2015

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import sys
from pathlib import Path

# Add our stats library to path
sys.path.append('../lib')
from stats_core import (
    calculate_temporal_change, 
    EURO_ADOPTION_DATES, 
    CRISIS_YEARS,
    get_significance_stars
)

# Setup paths
data_dir = Path('../data')
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

print("CS2 Baltic Euro Adoption Analysis")
print("="*50)
print(f"Euro Adoption Dates:")
for country, year in EURO_ADOPTION_DATES.items():
    print(f"  {country}: {year}")
print()

## 2. Load and Prepare Data

In [ ]:
# Load comprehensive dataset
print("Loading comprehensive dataset...")
df = pd.read_csv(data_dir / 'comprehensive_df_PGDP_labeled.csv')

# Filter for Baltic countries
baltic_countries = list(EURO_ADOPTION_DATES.keys())
baltic_data = df[df['COUNTRY'].isin(baltic_countries)].copy()

print(f"Data shape after filtering for Baltic countries: {baltic_data.shape}")
print(f"Countries: {baltic_data['COUNTRY'].unique().tolist()}")
print(f"Years: {baltic_data['YEAR'].min()} to {baltic_data['YEAR'].max()}")
print()

## 3. Define Analysis Indicators

In [ ]:
# Define indicators to analyze (PGDP suffix for % of GDP normalization)
indicators = [
    'Direct_Investment_PGDP',
    'Portfolio_Investment_PGDP',
    'Financial_Derivatives_PGDP',
    'Other_Investment_PGDP',
    'Portfolio_Debt_Securities_PGDP',
    'Portfolio_Equity_PGDP',
    'Capital_Account_PGDP',
    'Net_Capital_Flows_PGDP',
    'Other_Investment_Other_Equity_PGDP',
    'Other_Investment_Currency_Deposits_PGDP',
    'Other_Investment_Debt_Instruments_PGDP',
    'Other_Investment_Loans_PGDP',
    'Other_Investment_Trade_Credit_Advances_PGDP',
    'Net_Capital_Inflows_PGDP'
]

# Check which indicators are available
available_indicators = [col for col in indicators if col in baltic_data.columns]
print(f"Available indicators: {len(available_indicators)} of {len(indicators)}")
for ind in available_indicators:
    print(f"  - {ind}")

## 4. Temporal Period Creation Function

In [ ]:
def create_temporal_periods(data, country, adoption_year, exclude_crisis=False):
    """
    Create pre/post Euro adoption periods for a country.
    Source: Extracted from cs2_shared_functions.py lines 63-95
    
    Parameters:
    -----------
    data : pd.DataFrame
        Data for the country
    country : str
        Country name
    adoption_year : int
        Year of Euro adoption
    exclude_crisis : bool
        Whether to exclude crisis years
    
    Returns:
    --------
    pd.DataFrame
        Data with 'EURO_PERIOD' column added
    """
    country_data = data[data['COUNTRY'] == country].copy()
    
    # Define periods
    country_data['EURO_PERIOD'] = 'Unknown'
    
    if exclude_crisis:
        # Exclude crisis years from both periods
        all_crisis = CRISIS_YEARS['ALL']
        
        # Pre-Euro: Before adoption year, excluding crisis years
        pre_mask = (country_data['YEAR'] < adoption_year) & (~country_data['YEAR'].isin(all_crisis))
        # Post-Euro: From adoption year onward, excluding crisis years  
        post_mask = (country_data['YEAR'] >= adoption_year) & (~country_data['YEAR'].isin(all_crisis))
    else:
        # Full series - include all years
        # Pre-Euro: Before adoption year
        pre_mask = country_data['YEAR'] < adoption_year
        # Post-Euro: From adoption year onward
        post_mask = country_data['YEAR'] >= adoption_year
    
    country_data.loc[pre_mask, 'EURO_PERIOD'] = 'Pre-Euro'
    country_data.loc[post_mask, 'EURO_PERIOD'] = 'Post-Euro'
    
    # Print period summary
    pre_years = country_data[country_data['EURO_PERIOD'] == 'Pre-Euro']['YEAR'].unique()
    post_years = country_data[country_data['EURO_PERIOD'] == 'Post-Euro']['YEAR'].unique()
    
    print(f"\n{country} - {'Crisis Excluded' if exclude_crisis else 'Full Series'}:")
    print(f"  Pre-Euro: {pre_years.min() if len(pre_years) > 0 else 'N/A'} to "
          f"{pre_years.max() if len(pre_years) > 0 else 'N/A'} ({len(pre_years)} years)")
    print(f"  Post-Euro: {post_years.min() if len(post_years) > 0 else 'N/A'} to "
          f"{post_years.max() if len(post_years) > 0 else 'N/A'} ({len(post_years)} years)")
    
    return country_data

# Test the function
print("Testing temporal period creation:")
print("="*50)
for country, year in EURO_ADOPTION_DATES.items():
    _ = create_temporal_periods(baltic_data, country, year, exclude_crisis=False)

## 5. Temporal F-Test Implementation

In [ ]:
def perform_temporal_f_test(pre_data, post_data):
    """
    Perform F-test comparing pre/post Euro volatility.
    Source: Extracted from cs2_shared_functions.py lines 224-236
    
    Returns:
    --------
    dict with f_statistic, p_value, pre_variance, post_variance
    """
    # Clean data
    pre_clean = pd.Series(pre_data).dropna()
    post_clean = pd.Series(post_data).dropna()
    
    if len(pre_clean) < 2 or len(post_clean) < 2:
        return {
            'f_statistic': np.nan,
            'p_value': np.nan,
            'pre_variance': np.nan,
            'post_variance': np.nan,
            'pre_n': len(pre_clean),
            'post_n': len(post_clean),
            'insufficient_data': True
        }
    
    # Calculate variances (sample variance with ddof=1)
    var_pre = np.var(pre_clean, ddof=1)
    var_post = np.var(post_clean, ddof=1)
    
    # F-test
    if var_pre > 0 and var_post > 0:
        f_stat = var_pre / var_post
        df1 = len(pre_clean) - 1
        df2 = len(post_clean) - 1
        # Two-tailed p-value
        p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))
    else:
        f_stat = np.nan
        p_value = np.nan
    
    return {
        'f_statistic': f_stat,
        'p_value': p_value,
        'pre_variance': var_pre,
        'post_variance': var_post,
        'pre_n': len(pre_clean),
        'post_n': len(post_clean),
        'variance_ratio': var_post / var_pre if var_pre > 0 else np.nan,
        'volatility_decreased': var_post < var_pre if not np.isnan(var_post) else None
    }

# Test the function
print("Testing F-test implementation:")
print("="*50)

# Create sample data
np.random.seed(42)
sample_pre = np.random.normal(0, 2, 50)  # Higher variance
sample_post = np.random.normal(0, 1, 50)  # Lower variance

test_result = perform_temporal_f_test(sample_pre, sample_post)
print(f"Sample F-test Result:")
print(f"  Pre-variance: {test_result['pre_variance']:.4f}")
print(f"  Post-variance: {test_result['post_variance']:.4f}")
print(f"  F-statistic: {test_result['f_statistic']:.4f}")
print(f"  P-value: {test_result['p_value']:.4f}")
print(f"  Volatility decreased: {test_result['volatility_decreased']}")

## 6. Main Analysis - Full Series (Including Crisis Years)

In [ ]:
# Analyze each country
full_series_results = []

print("\n" + "="*80)
print("FULL SERIES ANALYSIS (INCLUDING CRISIS YEARS)")
print("="*80)

for country, adoption_year in EURO_ADOPTION_DATES.items():
    print(f"\n{country} (Euro adoption: {adoption_year})")
    print("-" * 60)
    
    # Create temporal periods
    country_data = create_temporal_periods(baltic_data, country, adoption_year, exclude_crisis=False)
    
    # Analyze each indicator
    for indicator in available_indicators:
        # Get pre and post data
        pre_data = country_data[country_data['EURO_PERIOD'] == 'Pre-Euro'][indicator]
        post_data = country_data[country_data['EURO_PERIOD'] == 'Post-Euro'][indicator]
        
        # Perform F-test
        result = perform_temporal_f_test(pre_data, post_data)
        
        # Get significance stars
        significance = get_significance_stars(result['p_value'])
        
        # Store results
        full_series_results.append({
            'Country': country,
            'Adoption_Year': adoption_year,
            'Indicator': indicator.replace('_PGDP', ''),
            'Pre_Variance': result['pre_variance'],
            'Post_Variance': result['post_variance'],
            'Variance_Ratio': result['variance_ratio'],
            'F_Statistic': result['f_statistic'],
            'P_Value': result['p_value'],
            'Significance': significance,
            'Volatility_Decreased': result['volatility_decreased'],
            'Pre_N': result['pre_n'],
            'Post_N': result['post_n'],
            'Series_Type': 'Full_Series'
        })
        
        # Print detailed results for significant changes
        if significance != '':
            print(f"\n  {indicator.replace('_PGDP', '')}:")
            print(f"    Pre-Euro variance: {result['pre_variance']:.6f}")
            print(f"    Post-Euro variance: {result['post_variance']:.6f}")
            print(f"    Variance ratio (post/pre): {result['variance_ratio']:.3f}")
            print(f"    F-statistic: {result['f_statistic']:.3f}")
            print(f"    P-value: {result['p_value']:.4f} {significance}")
            print(f"    Volatility {'DECREASED' if result['volatility_decreased'] else 'INCREASED'}")

# Convert to DataFrame
full_results_df = pd.DataFrame(full_series_results)
print(f"\n\nTotal tests performed: {len(full_results_df)}")
print(f"Significant at 5% level: {(full_results_df['P_Value'] < 0.05).sum()}")
print(f"Volatility decreased cases: {full_results_df['Volatility_Decreased'].sum()}")

## 7. Crisis-Excluded Analysis

In [ ]:
# Analyze excluding crisis years
crisis_excluded_results = []

print("\n" + "="*80)
print("CRISIS-EXCLUDED ANALYSIS (EXCLUDING GFC 2008-2010 & COVID 2020-2022)")
print("="*80)

for country, adoption_year in EURO_ADOPTION_DATES.items():
    print(f"\n{country} (Euro adoption: {adoption_year})")
    print("-" * 60)
    
    # Create temporal periods excluding crisis
    country_data = create_temporal_periods(baltic_data, country, adoption_year, exclude_crisis=True)
    
    # Analyze each indicator
    for indicator in available_indicators:
        # Get pre and post data
        pre_data = country_data[country_data['EURO_PERIOD'] == 'Pre-Euro'][indicator]
        post_data = country_data[country_data['EURO_PERIOD'] == 'Post-Euro'][indicator]
        
        # Perform F-test
        result = perform_temporal_f_test(pre_data, post_data)
        
        # Get significance stars
        significance = get_significance_stars(result['p_value'])
        
        # Store results
        crisis_excluded_results.append({
            'Country': country,
            'Adoption_Year': adoption_year,
            'Indicator': indicator.replace('_PGDP', ''),
            'Pre_Variance': result['pre_variance'],
            'Post_Variance': result['post_variance'],
            'Variance_Ratio': result['variance_ratio'],
            'F_Statistic': result['f_statistic'],
            'P_Value': result['p_value'],
            'Significance': significance,
            'Volatility_Decreased': result['volatility_decreased'],
            'Pre_N': result['pre_n'],
            'Post_N': result['post_n'],
            'Series_Type': 'Crisis_Excluded'
        })
        
        # Print detailed results for significant changes
        if significance != '':
            print(f"\n  {indicator.replace('_PGDP', '')}:")
            print(f"    Pre-Euro variance: {result['pre_variance']:.6f}")
            print(f"    Post-Euro variance: {result['post_variance']:.6f}")
            print(f"    Variance ratio (post/pre): {result['variance_ratio']:.3f}")
            print(f"    F-statistic: {result['f_statistic']:.3f}")
            print(f"    P-value: {result['p_value']:.4f} {significance}")
            print(f"    Volatility {'DECREASED' if result['volatility_decreased'] else 'INCREASED'}")

# Convert to DataFrame
crisis_excluded_df = pd.DataFrame(crisis_excluded_results)
print(f"\n\nTotal tests performed: {len(crisis_excluded_df)}")
print(f"Significant at 5% level: {(crisis_excluded_df['P_Value'] < 0.05).sum()}")
print(f"Volatility decreased cases: {crisis_excluded_df['Volatility_Decreased'].sum()}")

## 8. Summary Statistics by Country

In [ ]:
def summarize_country_results(results_df, country):
    """
    Summarize temporal volatility results for a specific country.
    """
    country_results = results_df[results_df['Country'] == country]
    
    # Count significant results
    total_indicators = len(country_results)
    significant_5pct = (country_results['P_Value'] < 0.05).sum()
    significant_10pct = (country_results['P_Value'] < 0.10).sum()
    volatility_decreased = country_results['Volatility_Decreased'].sum()
    
    # Average variance change
    avg_variance_ratio = country_results['Variance_Ratio'].mean()
    
    return {
        'Country': country,
        'Total_Indicators': total_indicators,
        'Significant_5pct': significant_5pct,
        'Significant_10pct': significant_10pct,
        'Volatility_Decreased': volatility_decreased,
        'Avg_Variance_Ratio': avg_variance_ratio,
        'Pct_Decreased': (volatility_decreased / total_indicators * 100) if total_indicators > 0 else 0
    }

print("\n" + "="*80)
print("SUMMARY BY COUNTRY")
print("="*80)

# Full series summary
print("\nFull Series (Including Crisis Years):")
print("-" * 50)
full_summaries = []
for country in EURO_ADOPTION_DATES.keys():
    summary = summarize_country_results(full_results_df, country)
    full_summaries.append(summary)
    print(f"\n{country}:")
    print(f"  Indicators analyzed: {summary['Total_Indicators']}")
    print(f"  Significant at 5%: {summary['Significant_5pct']}")
    print(f"  Significant at 10%: {summary['Significant_10pct']}")
    print(f"  Volatility decreased: {summary['Volatility_Decreased']} ({summary['Pct_Decreased']:.1f}%)")
    print(f"  Average variance ratio: {summary['Avg_Variance_Ratio']:.3f}")

# Crisis-excluded summary
print("\n\nCrisis-Excluded Series:")
print("-" * 50)
crisis_summaries = []
for country in EURO_ADOPTION_DATES.keys():
    summary = summarize_country_results(crisis_excluded_df, country)
    crisis_summaries.append(summary)
    print(f"\n{country}:")
    print(f"  Indicators analyzed: {summary['Total_Indicators']}")
    print(f"  Significant at 5%: {summary['Significant_5pct']}")
    print(f"  Significant at 10%: {summary['Significant_10pct']}")
    print(f"  Volatility decreased: {summary['Volatility_Decreased']} ({summary['Pct_Decreased']:.1f}%)")
    print(f"  Average variance ratio: {summary['Avg_Variance_Ratio']:.3f}")

## 9. Compare Full vs Crisis-Excluded Results

In [ ]:
print("\n" + "="*80)
print("COMPARISON: FULL SERIES vs CRISIS-EXCLUDED")
print("="*80)

for country in EURO_ADOPTION_DATES.keys():
    print(f"\n{country}:")
    print("-" * 40)
    
    # Get both sets of results
    full_country = full_results_df[full_results_df['Country'] == country]
    crisis_country = crisis_excluded_df[crisis_excluded_df['Country'] == country]
    
    # Merge on indicator to compare
    comparison = pd.merge(
        full_country[['Indicator', 'Variance_Ratio', 'P_Value', 'Volatility_Decreased']],
        crisis_country[['Indicator', 'Variance_Ratio', 'P_Value', 'Volatility_Decreased']],
        on='Indicator',
        suffixes=('_Full', '_Crisis_Excl')
    )
    
    # Find indicators with different conclusions
    different_significance = comparison[
        ((comparison['P_Value_Full'] < 0.05) != (comparison['P_Value_Crisis_Excl'] < 0.05))
    ]
    
    different_direction = comparison[
        (comparison['Volatility_Decreased_Full'] != comparison['Volatility_Decreased_Crisis_Excl'])
    ]
    
    print(f"\n  Indicators with different significance (5% level): {len(different_significance)}")
    if len(different_significance) > 0:
        for _, row in different_significance.iterrows():
            full_sig = "significant" if row['P_Value_Full'] < 0.05 else "not significant"
            crisis_sig = "significant" if row['P_Value_Crisis_Excl'] < 0.05 else "not significant"
            print(f"    - {row['Indicator']}: Full={full_sig}, Crisis-Excl={crisis_sig}")
    
    print(f"\n  Indicators with different volatility direction: {len(different_direction)}")
    if len(different_direction) > 0:
        for _, row in different_direction.iterrows():
            full_dir = "decreased" if row['Volatility_Decreased_Full'] else "increased"
            crisis_dir = "decreased" if row['Volatility_Decreased_Crisis_Excl'] else "increased"
            print(f"    - {row['Indicator']}: Full={full_dir}, Crisis-Excl={crisis_dir}")

## 10. Save Results

In [ ]:
# Save detailed results
full_results_df.to_csv(output_dir / 'CS2_full_series_results.csv', index=False)
crisis_excluded_df.to_csv(output_dir / 'CS2_crisis_excluded_results.csv', index=False)

# Save country summaries
pd.DataFrame(full_summaries).to_csv(output_dir / 'CS2_country_summary_full.csv', index=False)
pd.DataFrame(crisis_summaries).to_csv(output_dir / 'CS2_country_summary_crisis_excluded.csv', index=False)

print("Results saved to:")
print(f"  - {output_dir / 'CS2_full_series_results.csv'}")
print(f"  - {output_dir / 'CS2_crisis_excluded_results.csv'}")
print(f"  - {output_dir / 'CS2_country_summary_full.csv'}")
print(f"  - {output_dir / 'CS2_country_summary_crisis_excluded.csv'}")

## 11. Verify Against Baseline (If Available)

In [ ]:
# Check if we have baseline results to compare
baseline_path = Path('../verification/baseline_results/CS2_baseline.csv')

if baseline_path.exists():
    print("\n" + "="*80)
    print("BASELINE VERIFICATION")
    print("="*80)
    
    baseline_df = pd.read_csv(baseline_path)
    print(f"\nBaseline data shape: {baseline_df.shape}")
    print(f"Baseline columns: {baseline_df.columns.tolist()}")
    
    # Compare with our results
    # Note: Exact comparison depends on baseline format
    print("\n[Baseline comparison would go here if format matches]")
else:
    print("\nNo baseline results found for CS2.")
    print("This is expected as CS2 baseline extraction is more complex due to temporal analysis.")

## 12. Key Findings Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# Overall statistics
total_tests = len(full_results_df)
total_decreased_full = full_results_df['Volatility_Decreased'].sum()
total_decreased_crisis = crisis_excluded_df['Volatility_Decreased'].sum()

print(f"\n1. OVERALL VOLATILITY CHANGES:")
print(f"   Full series: {total_decreased_full}/{total_tests} indicators showed decreased volatility ({total_decreased_full/total_tests*100:.1f}%)")
print(f"   Crisis-excluded: {total_decreased_crisis}/{total_tests} indicators showed decreased volatility ({total_decreased_crisis/total_tests*100:.1f}%)")

print(f"\n2. STATISTICAL SIGNIFICANCE (at 5% level):")
for country in EURO_ADOPTION_DATES.keys():
    full_sig = (full_results_df[full_results_df['Country'] == country]['P_Value'] < 0.05).sum()
    crisis_sig = (crisis_excluded_df[crisis_excluded_df['Country'] == country]['P_Value'] < 0.05).sum()
    print(f"   {country}: Full={full_sig}, Crisis-excluded={crisis_sig}")

print(f"\n3. AVERAGE VARIANCE RATIOS (Post/Pre):")
for country in EURO_ADOPTION_DATES.keys():
    full_avg = full_results_df[full_results_df['Country'] == country]['Variance_Ratio'].mean()
    crisis_avg = crisis_excluded_df[crisis_excluded_df['Country'] == country]['Variance_Ratio'].mean()
    print(f"   {country}: Full={full_avg:.3f}, Crisis-excluded={crisis_avg:.3f}")
    if full_avg < 1:
        print(f"     → Overall volatility decreased after Euro adoption")
    else:
        print(f"     → Overall volatility increased after Euro adoption")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)